# Standardised Data Cleaning Notebook

This notebook **standardises** the **data cleaning** process for **historical stablecoin datasets**. Users should **update** the file paths for both *data loading and saving* accordingly. The cleaned data is saved in **Parquet** format to *preserve data types*. This workflow can be easily replicated across other stablecoin datasets by modifying the file paths.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Data Cleaning

In [2]:
# load data
stablecoin = "dai" # or "usdc", "usdt", "pax", "dai", "ust"
path = f"raw_data/{stablecoin}_hist_data.csv"
df = pd.read_csv(path)
# for ust only 
# df = df[df["timestamp"] <= "2022-05-11"].copy()

In [3]:
# parse datetimes + numerics
time_cols = ["timestamp", "timeOpen", "timeClose", "timeHigh", "timeLow"]
for c in time_cols:
    df[c] = pd.to_datetime(df[c])

num_cols = ["open", "high", "low", "close", "volume", "marketCap", "circulatingSupply"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c])

df.dtypes

symbol                       object
timestamp            datetime64[ns]
timeOpen             datetime64[ns]
timeClose            datetime64[ns]
timeHigh             datetime64[ns]
timeLow              datetime64[ns]
open                        float64
high                        float64
low                         float64
close                       float64
volume                      float64
marketCap                   float64
circulatingSupply           float64
dtype: object

In [4]:
# drop duplicates (same timestamp)
print(f"Number of rows before dropping duplicates: {len(df)}")
df = df.sort_values("timestamp")
df = df.drop_duplicates(subset=["timestamp"], keep="last")
print(f"Number of rows after dropping duplicates: {len(df)}")

Number of rows before dropping duplicates: 1941
Number of rows after dropping duplicates: 1941


### Basic Price Sanity Checks

In [5]:
# ensure low > 0, high > 0, and low <= high for all observations
# if any violations exist, they should be cleaned before analysis

invalid_low = (df["low"] <= 0).sum()
invalid_high = (df["high"] <= 0).sum()
invalid_order = (df["low"] > df["high"]).sum()

print(invalid_low, invalid_high, invalid_order)

# in this dataset, all values are valid (all counts = 0),
# so no additional cleaning is required for low/high prices.

0 0 0


## Depeg Labelling

In [6]:
deviation = {"ust": 0.006, "usdc": 0.001, "usdt": 0.002, "pax": 0.007, "dai": 0.004}

# create current depeg label
df["depeg"] = (abs(df["close"] - 1) >= deviation[stablecoin]).astype(int)
# check number of depegs
df["depeg"].sum()

32

In [7]:
df.head(10)

,symbol,timestamp,timeOpen,timeClose,timeHigh,timeLow,open,high,low,close,volume,marketCap,circulatingSupply,depeg
0,DAI,2020-11-25 23:59:59,2020-11-25,2020-11-25 23:59:59,2020-11-25 07:25:07,2020-11-25 01:50:07,1.001130,1.002827,0.999859,1.001239,1.359826e+08,1.046332e+09,1.045037e+09,0
1,DAI,2020-11-26 23:59:59,2020-11-26,2020-11-26 23:59:59,2020-11-26 09:03:06,2020-11-26 04:08:06,1.001180,1.020000,1.000657,1.004435,3.389927e+08,1.016911e+09,1.012421e+09,1
2,DAI,2020-11-27 23:59:59,2020-11-27,2020-11-27 23:59:59,2020-11-27 15:58:06,2020-11-27 09:21:07,1.004195,1.005209,1.001301,1.002698,9.298694e+07,1.017528e+09,1.014791e+09,0
3,DAI,2020-11-28 23:59:59,2020-11-28,2020-11-28 23:59:59,2020-11-28 09:25:08,2020-11-28 22:13:07,1.002822,1.005799,1.001009,1.002362,8.283827e+07,1.019335e+09,1.016932e+09,0
4,DAI,2020-11-29 23:59:59,2020-11-29,2020-11-29 23:59:59,2020-11-29 02:05:07,2020-11-29 00:07:07,1.002233,1.005793,1.001375,1.003909,8.097276e+07,1.033592e+09,1.029567e+09,0
5,DAI,2020-11-30 23:59:59,2020-11-30,2020-11-30 23:59:59,2020-11-30 19:22:06,2020-11-30 08:39:07,1.004047,1.007034,1.003216,1.006447,1.631315e+08,1.053445e+09,1.046697e+09,1
6,DAI,2020-12-01 23:59:59,2020-12-01,2020-12-01 23:59:59,2020-12-01 13:24:07,2020-12-01 18:28:07,1.006637,1.008279,1.004205,1.004682,1.694766e+08,1.062265e+09,1.057314e+09,1
7,DAI,2020-12-02 23:59:59,2020-12-02,2020-12-02 23:59:59,2020-12-02 12:24:07,2020-12-02 17:54:07,1.004911,1.005746,1.003107,1.003746,1.056988e+08,1.076028e+09,1.072012e+09,0
8,DAI,2020-12-03 23:59:59,2020-12-03,2020-12-03 23:59:59,2020-12-03 11:01:07,2020-12-03 07:21:07,1.003775,1.005375,1.002967,1.003665,9.176070e+07,1.080757e+09,1.076811e+09,0
9,DAI,2020-12-04 23:59:59,2020-12-04,2020-12-04 23:59:59,2020-12-04 04:39:07,2020-12-04 13:13:07,1.003706,1.004773,1.002364,1.003605,1.141819e+08,1.073433e+09,1.069577e+09,0


In [8]:
# create future depeg labels (within next h days)
horizons = [1, 3, 5, 7, 14, 30]

for h in horizons:
    df[f"depeg_future_{h}d"] = (
        df["depeg"]
        .shift(-1) # start from tomorrow
        .rolling(window=h, min_periods=1)
        .max()
        .astype("float") # keep NaN at tail first
    )

# handle tail NaNs (no future data to check, so assume no depeg)
for h in horizons:
    col = f"depeg_future_{h}d"
    df[col] = df[col].fillna(0).astype(int)

df.head()

,symbol,timestamp,timeOpen,timeClose,timeHigh,timeLow,open,high,low,close,volume,marketCap,circulatingSupply,depeg,depeg_future_1d,depeg_future_3d,depeg_future_5d,depeg_future_7d,depeg_future_14d,depeg_future_30d
0,DAI,2020-11-25 23:59:59,2020-11-25,2020-11-25 23:59:59,2020-11-25 07:25:07,2020-11-25 01:50:07,1.001130,1.002827,0.999859,1.001239,1.359826e+08,1.046332e+09,1.045037e+09,0,1,1,1,1,1,1
1,DAI,2020-11-26 23:59:59,2020-11-26,2020-11-26 23:59:59,2020-11-26 09:03:06,2020-11-26 04:08:06,1.001180,1.020000,1.000657,1.004435,3.389927e+08,1.016911e+09,1.012421e+09,1,0,1,1,1,1,1
2,DAI,2020-11-27 23:59:59,2020-11-27,2020-11-27 23:59:59,2020-11-27 15:58:06,2020-11-27 09:21:07,1.004195,1.005209,1.001301,1.002698,9.298694e+07,1.017528e+09,1.014791e+09,0,0,1,1,1,1,1
3,DAI,2020-11-28 23:59:59,2020-11-28,2020-11-28 23:59:59,2020-11-28 09:25:08,2020-11-28 22:13:07,1.002822,1.005799,1.001009,1.002362,8.283827e+07,1.019335e+09,1.016932e+09,0,0,0,1,1,1,1
4,DAI,2020-11-29 23:59:59,2020-11-29,2020-11-29 23:59:59,2020-11-29 02:05:07,2020-11-29 00:07:07,1.002233,1.005793,1.001375,1.003909,8.097276e+07,1.033592e+09,1.029567e+09,0,1,1,1,1,1,1


In [9]:
# reorder columns (place labels after first column) ---
cols = df.columns.tolist()

first_col = cols[0]
label_cols = ["depeg"] + [f"depeg_future_{h}d" for h in horizons]

# remove label cols first
remaining_cols = [c for c in cols if c not in label_cols]

# rebuild column order
new_cols = [first_col] + label_cols + [c for c in remaining_cols if c != first_col]

df = df[new_cols]

df.head()

,symbol,depeg,depeg_future_1d,depeg_future_3d,depeg_future_5d,depeg_future_7d,depeg_future_14d,depeg_future_30d,timestamp,timeOpen,timeClose,timeHigh,timeLow,open,high,low,close,volume,marketCap,circulatingSupply
0,DAI,0,1,1,1,1,1,1,2020-11-25 23:59:59,2020-11-25,2020-11-25 23:59:59,2020-11-25 07:25:07,2020-11-25 01:50:07,1.001130,1.002827,0.999859,1.001239,1.359826e+08,1.046332e+09,1.045037e+09
1,DAI,1,0,1,1,1,1,1,2020-11-26 23:59:59,2020-11-26,2020-11-26 23:59:59,2020-11-26 09:03:06,2020-11-26 04:08:06,1.001180,1.020000,1.000657,1.004435,3.389927e+08,1.016911e+09,1.012421e+09
2,DAI,0,0,1,1,1,1,1,2020-11-27 23:59:59,2020-11-27,2020-11-27 23:59:59,2020-11-27 15:58:06,2020-11-27 09:21:07,1.004195,1.005209,1.001301,1.002698,9.298694e+07,1.017528e+09,1.014791e+09
3,DAI,0,0,0,1,1,1,1,2020-11-28 23:59:59,2020-11-28,2020-11-28 23:59:59,2020-11-28 09:25:08,2020-11-28 22:13:07,1.002822,1.005799,1.001009,1.002362,8.283827e+07,1.019335e+09,1.016932e+09
4,DAI,0,1,1,1,1,1,1,2020-11-29 23:59:59,2020-11-29,2020-11-29 23:59:59,2020-11-29 02:05:07,2020-11-29 00:07:07,1.002233,1.005793,1.001375,1.003909,8.097276e+07,1.033592e+09,1.029567e+09


---

## Save

In [10]:
df.dtypes

symbol                       object
depeg                         int64
depeg_future_1d               int64
depeg_future_3d               int64
depeg_future_5d               int64
depeg_future_7d               int64
depeg_future_14d              int64
depeg_future_30d              int64
timestamp            datetime64[ns]
timeOpen             datetime64[ns]
timeClose            datetime64[ns]
timeHigh             datetime64[ns]
timeLow              datetime64[ns]
open                        float64
high                        float64
low                         float64
close                       float64
volume                      float64
marketCap                   float64
circulatingSupply           float64
dtype: object

In [11]:
df.to_parquet(f"clean_data/{stablecoin}_clean.parquet")